# Notebook 01: Setup and Baseline

Loads Gemma 2 2B (or GPT-2 Small fallback) and the corresponding Gemma Scope SAE.
Records baseline metrics: model perplexity, SAE L0, FVU, and next-token accuracy.
All results are saved to `results/exp0_baseline.json`.

In [1]:
# Install all dependencies (Colab cell — run first)
!pip install sae-lens==4.3.1 transformers==4.44.2 \
    datasets==2.20.0 tqdm==4.66.4 scipy==1.13.1 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 911.2/911.2 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os, sys

# ── UPDATE THIS to your actual GitHub repo URL before running ──────────────────────
GITHUB_REPO_URL = "https://github.com/rajo69/Adversarial-Robustness-of-SAE-Interpretations"
# ───────────────────────────────────────────────────────────────────────────

REPO_DIR = "/content/SAE_Adversarial_Project"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {REPO_DIR}")

Working directory: /content/SAE_Adversarial_Project
Python path includes: /content/SAE_Adversarial_Project


In [3]:
# Optional: mount Google Drive to persist results across sessions
# Uncomment to enable.
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_RESULTS = "/content/drive/MyDrive/sae_adversarial/results"
# DRIVE_FIGURES = "/content/drive/MyDrive/sae_adversarial/figures"
# os.makedirs(DRIVE_RESULTS, exist_ok=True)
# os.makedirs(DRIVE_FIGURES, exist_ok=True)
# print("Drive mounted. Results will be saved to Drive in addition to local.")

In [4]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Fixed seed everywhere
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
from pathlib import Path

RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
print(f"Results \u2192 {RESULTS_DIR.resolve()}")
print(f"Figures \u2192 {FIGURES_DIR.resolve()}")

In [ ]:
from transformer_lens import HookedTransformer

# Model configuration — auto-selected based on what fits in VRAM
# Gemma 2 2B needs ~10 GB; GPT-2 Small needs < 1 GB.
try:
    print("Loading Gemma 2 2B \u2026")
    t0 = time.time()
    model = HookedTransformer.from_pretrained("gemma-2-2b", device=DEVICE)
    model.eval()
    MODEL_NAME = "gemma-2-2b"
    TOKENIZER_NAME = "google/gemma-2-2b"
    SAE_RELEASE = "gemma-scope-2b-pt-res"
    # SAE ID template: fill in {layer} for each layer
    SAE_ID_TEMPLATE = "layer_{layer}/width_16k/average_l0_82"
    TARGET_LAYER = 12           # middle layer; change after OOM check
    HOOK_TEMPLATE = "blocks.{layer}.hook_resid_post"
    N_LAYERS = 26
    print(f"Gemma 2 2B loaded in {time.time()-t0:.1f}s")
except Exception as e:
    print(f"Gemma 2 2B failed ({e}), falling back to GPT-2 Small \u2026")
    model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    model.eval()
    MODEL_NAME = "gpt2"
    TOKENIZER_NAME = "gpt2"
    SAE_RELEASE = "gpt2-small-res-jb"
    SAE_ID_TEMPLATE = "blocks.{layer}.hook_resid_pre"
    TARGET_LAYER = 8
    HOOK_TEMPLATE = "blocks.{layer}.hook_resid_pre"
    N_LAYERS = 12
    print("GPT-2 Small loaded.")

print(f"\nModel: {MODEL_NAME} | Target layer: {TARGET_LAYER}")
if DEVICE == "cuda":
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory \u2014 allocated: {alloc:.2f} GB | reserved: {reserved:.2f} GB")

In [ ]:
from sae_lens import SAE

print(f"Loading SAE for layer {TARGET_LAYER} \u2026")
t0 = time.time()
sae_id = SAE_ID_TEMPLATE.format(layer=TARGET_LAYER)
sae, sae_cfg, sae_sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=sae_id,
)
sae = sae.to(DEVICE)
sae.eval()
print(f"SAE loaded in {time.time()-t0:.1f}s")
print(f"  Release:    {SAE_RELEASE}")
print(f"  SAE id:     {sae_id}")
print(f"  SAE width:  {sae_cfg.get('d_sae', 'unknown')}")
if DEVICE == "cuda":
    alloc = torch.cuda.memory_allocated() / 1e9
    print(f"  GPU after:  {alloc:.2f} GB allocated")

In [ ]:
from src.data import prepare_dataset, tokens_to_tensor

print("Building evaluation set (WikiText-2 test, 150 prompts, seed=42) \u2026")
eval_set = prepare_dataset(
    model_name=TOKENIZER_NAME,
    n_prompts=150,
    min_tokens=64,
    max_tokens=128,
    seed=SEED,
    save=True,
    force_rebuild=False,
)
debug_set = eval_set[:10]
print(f"Eval set: {len(eval_set)} prompts | Debug set: {len(debug_set)} prompts")
token_lengths = [item["n_tokens"] for item in eval_set]
print(f"Token lengths \u2014 min: {min(token_lengths)}  max: {max(token_lengths)}  "
      f"mean: {np.mean(token_lengths):.1f}  std: {np.std(token_lengths):.1f}")

In [ ]:
# Verify model + SAE pipeline on one prompt before running full eval
sample = eval_set[0]
tokens = tokens_to_tensor(sample["tokens"], device=DEVICE)
hook_name = HOOK_TEMPLATE.format(layer=TARGET_LAYER)

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens, names_filter=hook_name)
    resid = cache[hook_name]                   # [1, seq_len, d_model]
    sae_acts = sae.encode(resid)               # [1, seq_len, sae_width]
    sae_recon = sae.decode(sae_acts)           # [1, seq_len, d_model] -- reconstruct

print(f"Token shape:    {tokens.shape}")
print(f"Resid shape:    {resid.shape}")
print(f"SAE acts shape: {sae_acts.shape}")
print(f"Top-1 predicted token: '{model.tokenizer.decode([logits[0, -1].argmax().item()])}'")

l0 = (sae_acts > 0).float().sum(dim=-1).mean().item()
fvu = ((resid - sae_recon)**2).mean().item() / resid.var().item()
print(f"SAE L0 (mean active features): {l0:.1f}")
print(f"SAE FVU (fraction variance unexplained): {fvu:.4f}")
torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
from src.eval_utils import compute_perplexity, next_token_accuracy, log_gpu_memory

log_gpu_memory("before baseline loop")
t0 = time.time()

perplexities = []
next_tok_accs = []
l0_values = []
fvu_values = []

for item in tqdm(eval_set, desc="Baseline eval"):
    tokens = tokens_to_tensor(item["tokens"], device=DEVICE)
    hook_name = HOOK_TEMPLATE.format(layer=TARGET_LAYER)

    with torch.no_grad():
        # Perplexity and accuracy
        ppl = compute_perplexity(model, tokens, device=DEVICE)
        acc = next_token_accuracy(model, tokens, device=DEVICE)
        perplexities.append(ppl)
        next_tok_accs.append(acc)

        # SAE L0 and FVU
        logits, cache = model.run_with_cache(tokens, names_filter=hook_name)
        resid = cache[hook_name]
        acts = sae.encode(resid)
        recon = sae.decode(acts)

        l0 = (acts > 0).float().sum(dim=-1).mean().item()
        fvu = ((resid - recon)**2).mean().item() / (resid.var().item() + 1e-10)
        l0_values.append(l0)
        fvu_values.append(fvu)

elapsed = time.time() - t0
log_gpu_memory("after baseline loop")

print(f"\nBaseline results ({len(eval_set)} prompts, {elapsed:.1f}s):")
print(f"  Perplexity:   {np.mean(perplexities):.2f} \u00b1 {np.std(perplexities):.2f}")
print(f"  Next-tok acc: {np.mean(next_tok_accs):.3f} \u00b1 {np.std(next_tok_accs):.3f}")
print(f"  SAE L0:       {np.mean(l0_values):.1f} \u00b1 {np.std(l0_values):.1f}")
print(f"  SAE FVU:      {np.mean(fvu_values):.4f} \u00b1 {np.std(fvu_values):.4f}")

In [ ]:
baseline_results = {
    "model": MODEL_NAME,
    "sae_release": SAE_RELEASE,
    "sae_id": SAE_ID_TEMPLATE.format(layer=TARGET_LAYER),
    "target_layer": TARGET_LAYER,
    "n_prompts": len(eval_set),
    "seed": SEED,
    "runtime_seconds": elapsed,
    "perplexity": {
        "mean": float(np.mean(perplexities)),
        "std":  float(np.std(perplexities)),
        "min":  float(np.min(perplexities)),
        "max":  float(np.max(perplexities)),
        "values": [float(x) for x in perplexities],
    },
    "next_token_accuracy": {
        "mean": float(np.mean(next_tok_accs)),
        "std":  float(np.std(next_tok_accs)),
        "values": [float(x) for x in next_tok_accs],
    },
    "sae_l0": {
        "mean": float(np.mean(l0_values)),
        "std":  float(np.std(l0_values)),
        "values": [float(x) for x in l0_values],
    },
    "sae_fvu": {
        "mean": float(np.mean(fvu_values)),
        "std":  float(np.std(fvu_values)),
        "values": [float(x) for x in fvu_values],
    },
}

out_path = RESULTS_DIR / "exp0_baseline.json"
with open(out_path, "w") as f:
    json.dump(baseline_results, f, indent=2)
print(f"Saved: {out_path}")

In [ ]:
from src.plot_config import apply_style, save_fig, COLORS

apply_style()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(perplexities, bins=30, color=COLORS["clean"], edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Perplexity")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Perplexity Distribution\n({MODEL_NAME}, n={len(eval_set)})")
axes[0].axvline(np.mean(perplexities), color=COLORS["adversarial"], linestyle="--",
                label=f"mean={np.mean(perplexities):.1f}")
axes[0].legend()

axes[1].hist(l0_values, bins=30, color=COLORS["random"], edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("SAE L0 (active features per token)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"SAE Sparsity Distribution\nLayer {TARGET_LAYER}")
axes[1].axvline(np.mean(l0_values), color=COLORS["adversarial"], linestyle="--",
                label=f"mean={np.mean(l0_values):.1f}")
axes[1].legend()

plt.tight_layout()
fig_path = FIGURES_DIR / "baseline_distributions.png"
save_fig(fig, str(fig_path))
plt.show()
print(f"Saved: {fig_path}")

## Baseline Complete

The baseline is established. Key numbers to record in `progress.md`:
- Model perplexity (mean ± std)
- SAE L0 (mean active features per token)
- SAE FVU (reconstruction quality)
- GPU memory usage before/after
- Runtime

Proceed to `02_feature_stability.ipynb` for Experiment 1.